# 01 - Introduction to LLM

Large Language Models (LLMs) are a type of AI models trained on vast amounts of text data.  

They work by predicting the next word (or token) in a sequence, which enables them to generate human-like language across a wide range of tasks,   
such as: Answering questions, summarisation, translation, code generation etc

At their core, LLMs don’t “know” facts in the traditional sense.  
They generate responses based on patterns learned during training.

This means they do not truly understand information or reason like humans. They simply rely on statistical patterns.

This leads to two key limitations:

- Non-deterministic behaviour with no guarantes of correctness.   
The same prompt can produce different responses depending on settings. The models also tend to hallucinate when it doesn't have the right answer.

- No real-time knowledge by default.  
They only generate what is in the model's training data unless additional context is provided.

But let's start with traditional approach

## The Traditional Approach: Rules-based

Before LLMs, solving natural language tasks required explicit rules for every scenario.

Example task: Recommend travel activities based on user preferences.

In [ ]:
# Traditional rules-based approach (rigid, limited, needs constant updates)
def recommend_activities_rules(preferences: str) -> str:
    """Recommend activities using hand-coded rules."""
    prefs_lower = preferences.lower()
    recommendations = []
    
    # Hard-coded keyword matching
    if "art" in prefs_lower or "museum" in prefs_lower:
        recommendations.append("Visit the Rijksmuseum")
    if "food" in prefs_lower or "eat" in prefs_lower:
        recommendations.append("Try stroopwafels at Albert Cuyp Market")
    if "history" in prefs_lower:
        recommendations.append("Visit Anne Frank House")
    
    if not recommendations:
        recommendations.append("Take a canal cruise")  # Default fallback
    
    return "\n".join(f"• {r}" for r in recommendations)


In [ ]:
test_preferences = [
    "I love art and museums",
    "Looking for good food spots",
    "I want something romantic for my anniversary",  # <-- No matching keywords!
    "Interested in Dutch culture and local experiences",  # <-- Too vague for rules
]

print("=== Rules-based Recommendations ===")
for pref in test_preferences:
    print(f"\nPreferences: '{pref}'")
    print(recommend_activities_rules(pref))
    print("------"*30)

**Notice the limitations:**
- "romantic anniversary" → Falls back to generic canal cruise (no keyword match)
- "Dutch culture" → Also falls back (too vague for our rules)

It is not feasible to add hundreds of keywords and rules to handle all possible requests. This is where LLMs can help!

## The New Approach: Generative

#### LLMs predict the next token

As said before, LLM reads the input text and predicts what is likely to come next.

It does this one piece at a time and those pieces are called **tokens**.

For example, if the model sees:

> The capital of France is

A likely next token sequence is:

> Paris

This simple idea is what's used to genearte explanations, summaries, code, plans, drafts, and conversations etc.

## Setup

#### Making the first API call


Before running the below cell, ensure you have:

1. Authenticated with `gcloud auth application-default login`
2. Set your GCP project and location below

The code below creates a genai client configured for Vertex AI.

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path().resolve().parents[1] / ".env")

In [ ]:
# This workshop uses the `google-genai` Python package with Vertex AI.
from google import genai

# Set GCP project and location
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
location = os.getenv("GOOGLE_CLOUD_LOCATION")

# Initialize the genai client for Vertex AI
client = genai.Client(
    vertexai=True, 
    project=project_id, 
    location=location
)

For simplicity, we'll define a helper function to call the LLM with our config

In [ ]:
from google.genai.types import GenerateContentConfig

# Select the model
MODEL_NAME = "gemini-2.5-flash"
# Set a default system message for the model
SYSTEM_MESSAGE = "You are a helpful assistant"


def ask_llm(
    prompt: str,
    system_instruction: str = SYSTEM_MESSAGE,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 1.0,
    # Add more parameters as needed
) -> str:
    """Send a prompt to the LLM and return the text response."""
    config = GenerateContentConfig(
        system_instruction=system_instruction,  # <-- Sets the model's persona/behavior for all responses
        temperature=temperature,                # <-- Controls randomness: 0=deterministic, 1=creative, 2=very random
        top_p=top_p,                            # <-- Nucleus sampling: considers tokens with cumulative probability up to this value
        top_k=top_k,                            # <-- Limits sampling to the top K most likely tokens at each step
    )
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )
    return response.text


## LLM Solves the Travel Problem

Now let's use the LLM to solve the same travel recommendation task that the rules-based approach struggled with.

Notice how we describe the behavior instead of coding explicit rules:

In [ ]:
# describe what you want in natural language
def recommend_activities_llm(preferences: str, **llm_kwargs) -> str:
    """Recommend activities using an LLM. Accepts extra LLM parameters as kwargs."""
    
    # Instead of coding rules, we describe the desired behavior
    system_instruction = """You are a travel guide for Amsterdam. 
    Based on the visitor's preferences, suggest 3 specific activities.
    Keep responses concise with bullet points.
    """
    
    # Pass all extra keyword arguments to ask_llm (e.g., temperature, top_k, top_p, etc.)
    return ask_llm(preferences, system_instruction=system_instruction, **llm_kwargs)

In [ ]:
test_preferences = [
    "I love art and museums",
    "Looking for good food spots",
    "I want something romantic for my anniversary",  # <-- Rules failed here!
    "Interested in Dutch culture and local experiences",  # <-- Rules failed here too!
]

print("=== LLM Recommendations ===")
for pref in test_preferences:
    print(f"\nPreferences: '{pref}'")
    print(recommend_activities_llm(pref))
    print("------"*30)

## Experiment: Vary Temperature, Top-K, and Top-P

LLMs can produce different valid answers to the same prompt. The parameters control how "creative" vs "focused" the output is.

Try adjusting these values on the travel recommendation and observe how the output changes:

#### LLM Sampling Settings and their effect

    | Task Type        | Temperature | Top-p     | Top-k   | Goal                                                                 |
    |------------------|-------------|-----------|---------|----------------------------------------------------------------------|
    | Coding / Math    | 0.0 – 0.2   | 0.1 – 0.3 | 1 – 10  | Precision. You want the single most likely answer.                   |
    | Fact Q&A         | 0.1 – 0.3   | 0.5       | 20      | Stability. Reduces hallucinations but allows natural phrasing.       |
    | General Chat     | 0.7         | 0.9       | 40      | The "Golden Ratio." Balanced, conversational, safe.                  |
    | Creative Writing | 0.8 – 1.0   | 0.95–0.99 | 50–100  | Surprise. Allows rarer words for flair.                              |

In [ ]:
preference = "I love art and museums. Suggest interesting activities."

# Deterministic
print("\nDeterministic:\n")
print(recommend_activities_llm(preference, temperature=0.0, top_k=1, top_p=0.1))
print("------"*10)


# Creative
print("\nCreative:\n")  
print(recommend_activities_llm(preference, temperature=1.8, top_k=64, top_p=1.0))
print("------"*10)


## Try Different Prompts

In [ ]:
# print(ask_llm("Explain LLMs like I am 12"))

In [ ]:
# print(ask_llm("Plan a 3-day trip to Tokyo"))

In [ ]:
# print(ask_llm("Give me 5 ideas for a weekend in Amsterdam"))